# Packages

In [11]:
from pathlib import Path
import scanpy as sc
import spatialdata as spd
from spatialdata.models import TableModel
from spatialdata import polygon_query

In [12]:
#Napari packages for tissue annotation
from qtpy.QtWidgets import QApplication
from napari_spatialdata import Interactive

In [51]:
import matplotlib.pyplot as plt
import spatialdata_plot  

# Functions

In [13]:
def make_sample_sdata(concat_sdata: spd.SpatialData, sample: str) -> spd.SpatialData:
    """
    Returns a SpatialData with:
      - the image for `sample`
      - the shapes for `sample` (cell boundaries)
      - the global table (segmentation_counts) unchanged
    """
    img_key = f"{sample}_hires_tissue_image"
    shp_key = f"{sample}_cell_boundaries"

    assert img_key in concat_sdata.images,  f"Missing image: {img_key}"
    assert shp_key in concat_sdata.shapes,  f"Missing shapes: {shp_key}"
    assert TABLE_KEY in concat_sdata.tables, "Missing table"

    sub = spd.SpatialData(
        images={img_key: concat_sdata.images[img_key]},
        shapes={shp_key: concat_sdata.shapes[shp_key]},
        tables={TABLE_KEY: concat_sdata.tables[TABLE_KEY]},
    )
    return sub


In [14]:
def crop_tissue(sub_sdata: spd.SpatialData, sample_id: str, tissue_name: str) -> spd.SpatialData:
    """
    sub_sdata: SpatialData for a single TMA (from make_sample_sdata)
    tissue_name: name of the ROI polygon in sub_sdata.shapes (e.g. "CyPSCA_1_1")

    Returns: SpatialData cropped to that polygon.
    """
    assert tissue_name in sub_sdata.shapes, f"ROI shape {tissue_name} not found in shapes."

    polygon = sub_sdata[tissue_name].geometry.iloc[0]
    cropped = polygon_query(
        sub_sdata,
        polygon=polygon,
        target_coordinate_system=CRS,
    )
    return cropped

In [24]:
def relabel_cropped_tissue(
    cropped_sdata: spd.SpatialData,
    sample_id: str,
    tissue_name: str,
) -> spd.SpatialData:
    """
    Take the cropped SpatialData (from crop_tissue) and:
      - rename image & shapes keys to tissue-specific names
      - add obs columns mouse, tissue
      - ensure region in the table matches the new shapes key
    """
    # Original keys in this cropped_sdata
    old_img_key = f"{sample_id}_hires_tissue_image"
    old_shp_key = f"{sample_id}_cell_boundaries"

    assert old_img_key in cropped_sdata.images, f"Expected image {old_img_key}"
    assert old_shp_key in cropped_sdata.shapes, f"Expected shapes {old_shp_key}"
    assert TABLE_KEY in cropped_sdata.tables, "Missing table in cropped_sdata"

    # New keys
    new_img_key = f"{tissue_name}_hires_tissue_image"
    new_shp_key = f"{tissue_name}_cell_boundaries"

    # Extract underlying data
    img_da      = cropped_sdata.images[old_img_key]
    shp_gdf     = cropped_sdata.shapes[old_shp_key]
    adata       = cropped_sdata.tables[TABLE_KEY].copy()

    # --- obs annotations ---
    
    adata.obs["TMA"]            = sample_id         # TMA ID (F07839, ...)
    adata.obs["mouse"]          = tissue_name       # the polygon / tumor / mouse label
    adata.obs["tissue"]         = tissue_name       # explicit tissue/ROI label
    adata.obs["condition"]      = tissue_name.split("_")[0] # Condition based on tissue name
    adata.obs["tumor_loc"]      = tissue_name.split("_")[1] # 1 = RT tumor; 2 = Abscopal tumor
    adata.obs["replicate_num"]  = tissue_name.split("_")[2] # Replicate number
    adata.obs["region"]         = new_shp_key
    adata.obs["region"]         = adata.obs["region"].astype("category")

    # Build a fresh SpatialData with tissue-level naming
    new_sdata = spd.SpatialData(
        images={new_img_key: img_da},
        shapes={
            new_shp_key: shp_gdf,
            # keep the ROI polygon as well, renamed explicitly as "_roi"
            f"{tissue_name}_roi": cropped_sdata.shapes[tissue_name],
        },
        tables={
            TABLE_KEY: TableModel.parse(
                adata,
                region=new_shp_key,
                region_key="region",
                instance_key="cell_id",
                overwrite_metadata=True,
            )
        },
    )

    return new_sdata


In [46]:
import matplotlib.pyplot as plt

def save_tissue_qc_plots(
    tissue_sd: spd.SpatialData,
    tissue_name: str,
    out_dir: Path,
) -> None:
    """
    For a tissue-level SpatialData (one image, one cell_boundaries, one ROI),
    save two sanity-check plots:
      1) H&E only
      2) H&E + cell boundaries overlay

    Files are saved in out_dir as:
      {tissue_name}_HE.png
      {tissue_name}_HE_cells.png
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # image key: first (and only) image
    img_key = list(tissue_sd.images.keys())[0]

    # shapes key: the one with '_cell_boundaries'
    shp_key_candidates = [k for k in tissue_sd.shapes.keys() if k.endswith("_cell_boundaries")]
    assert len(shp_key_candidates) == 1, f"Expected exactly one '*_cell_boundaries' in shapes, found {shp_key_candidates}"
    shp_key = shp_key_candidates[0]

    # --- Plot 1: H&E only ---
    fig, ax = plt.subplots(figsize=(6, 6))
    tissue_sd.pl.render_images(img_key).pl.show(
        coordinate_systems="downscale_to_hires",
        ax=ax,
    )
    ax.set_title(f"{tissue_name} - polygon_query H&E")
    he_path = out_dir / f"{tissue_name}_HE.png"
    fig.savefig(he_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # --- Plot 2: H&E + cell boundaries ---
    fig, ax = plt.subplots(figsize=(6, 6))
    (
        tissue_sd.pl.render_images(img_key)
        .pl.render_shapes(
            shp_key,
            color="lightgrey",
            fill_alpha=0.4,
            outline_alpha=0.4,
            outline_color="black",
            outline_width=0.2,
            method="matplotlib",
        )
        .pl.show(
            coordinate_systems="downscale_to_hires",
            ax=ax,
        )
    )
    ax.set_title(f"{tissue_name} - H&E + cell boundaries")
    he_cells_path = out_dir / f"{tissue_name}_HE_cells.png"
    fig.savefig(he_cells_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# Data Prep

In [16]:
# Main foldereed
zarr_folder = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles")
concatenated  = zarr_folder / "concatenated_sdata"

sdata = spd.read_zarr(concatenated)

# These shouldn't ever change, but can be edited if we get more complex datasets in the future
TABLE_KEY = "segmentation_counts"
CRS       = "downscale_to_hires"

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


## Pre define tissue

In [17]:
TMA_tissues = {
    "F07839" : ["CyPSCA_1_1", "RTCyPSCA_1_4", "NoTx_2_2", "RTCyT72_2_1"],
    "F07840" : ["CyPSCA_1_2", "RTCyPSCA_2_4", "CyT72_1_4", "RTCyT72_2_4"],
    "F08542" : ["NoTx_1_4", "CyT72_2_3", "CyPSCA_2_4", "RTCyPSCA_1_3"],
    "F08543" : ["NoTx_2_4", "CyT72_1_2", "RTCyT72_1_1", "RTCyPSCA_2_3"],
}

# Manual Step

## Sub-sampling

In [65]:
# Tissue already processed: 

# Tissue to process
sample_id = "F08543"
sub_sdata = make_sample_sdata(sdata, sample_id)

/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F07839_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F07840_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'F08542_cell_boundaries', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


## Napari

In [66]:
# Each shape needs to be saved one at a time
app = QApplication.instance() or QApplication([])
viewer = Interactive(sub_sdata)
 
app.exec()  # blocks until you close the napari window

2025-12-02 21:53:19.867 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2025-12-02 21:53:26.017 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:53:26.018 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:53:36.999 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:53:37.001 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:54:50.608 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually sa

INFO: Layer(s) inherited info from F08543_hires_tissue_image
INFO: Layer saved


2025-12-02 21:54:52.044 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:54:52.047 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:16.808 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:17.498 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:18.437 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:18.443 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:19.235 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:20.695 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:55:20.697 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:56:13.589 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updati

INFO: Layer(s) inherited info from F08543_hires_tissue_image
INFO: Layer saved


2025-12-02 21:57:05.411 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:57:05.414 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:59:14.837 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from F08543_hires_tissue_image
INFO: Layer saved


2025-12-02 21:59:23.138 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 21:59:23.140 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-12-02 22:00:16.285 | WARNING  | napari_spatialdata._viewer:_write_element_to_disk:178 - Annotations only added in memory, please manually save to disk.


INFO: Layer(s) inherited info from F08543_hires_tissue_image
INFO: Layer saved


0

In [67]:
sub_sdata

SpatialData object
├── Images
│     └── 'F08543_hires_tissue_image': DataArray[cyx] (3, 5652, 6000)
├── Shapes
│     ├── 'CyT72_1_2': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'F08543_cell_boundaries': GeoDataFrame shape: (58897, 2) (2D shapes)
│     ├── 'NoTx_2_4': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'RTCyPSCA_2_3': GeoDataFrame shape: (1, 1) (2D shapes)
│     └── 'RTCyT72_1_1': GeoDataFrame shape: (1, 1) (2D shapes)
└── Tables
      └── 'segmentation_counts': AnnData (428822, 19059)
with coordinate systems:
    ▸ 'downscale_to_hires', with elements:
        F08543_hires_tissue_image (Images), CyT72_1_2 (Shapes), F08543_cell_boundaries (Shapes), NoTx_2_4 (Shapes), RTCyPSCA_2_3 (Shapes), RTCyT72_1_1 (Shapes)

## Per TMA processing

In [69]:
tissue_sdatas_for_this_sample = []

for tissue_name in TMA_tissues[sample_id]:
    print(f"Processing {sample_id} / {tissue_name}")

    cropped = crop_tissue(sub_sdata, sample_id, tissue_name)
    tissue_sd = relabel_cropped_tissue(cropped, sample_id, tissue_name)

    # Sanity check: number of cells and new obs columns
    adata_tissue = tissue_sd.tables[TABLE_KEY]
    print(
        f"  -> cells: {adata_tissue.n_obs}, "
        f"mouse={adata_tissue.obs['mouse'].unique()}, "
        f"tissue={adata_tissue.obs['tissue'].unique()}"
    )

    tissue_sdatas_for_this_sample.append(tissue_sd)

Processing F08543 / NoTx_2_4
  -> cells: 31434, mouse=['NoTx_2_4'], tissue=['NoTx_2_4']
Processing F08543 / CyT72_1_2
  -> cells: 5242, mouse=['CyT72_1_2'], tissue=['CyT72_1_2']
Processing F08543 / RTCyT72_1_1
  -> cells: 2747, mouse=['RTCyT72_1_1'], tissue=['RTCyT72_1_1']
Processing F08543 / RTCyPSCA_2_3
  -> cells: 19117, mouse=['RTCyPSCA_2_3'], tissue=['RTCyPSCA_2_3']


# Saving

## Individual Tissues

In [70]:
# import shutil
tissue_out = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue")
tissue_out.mkdir(parents=True, exist_ok=True)

# directory for sanity-check images
tissue_img_out = tissue_out / "Tissue_images"
tissue_img_out.mkdir(parents=True, exist_ok=True)

for tissue_sd in tissue_sdatas_for_this_sample:
    # name by first (and only) tissue image key
    img_key = list(tissue_sd.images.keys())[0]
    tissue_name = img_key.replace("_hires_tissue_image", "")
    out_path = tissue_out / f"{sample_id}_{tissue_name}.zarr"
    print(f"Writing {out_path}")
    tissue_sd.write(out_path, overwrite=True)

    # save sanity-check plots for this tissue
    save_tissue_qc_plots(
        tissue_sd=tissue_sd,
        tissue_name=tissue_name,
        out_dir=tissue_img_out,
    )


Writing /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr
INFO     The SpatialData object is not self-contained (i.e. it contains some elements that are Dask-backed from    
         locations outside /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr).    
         Please see the documentation of `is_self_contained()` to understand the implications of working with      
         SpatialData objects that are not self-contained.                                                          
INFO     The Zarr backing store has been changed from None the new file path:                                      
         /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/ByTissue/F08543_NoTx_2_4.zarr                        
INFO     Rasterizing image for faster rendering.                                                                   
INFO     Value for parameter 'color' appears to be a color, using it as such.                    

## Complete Datasets concatenated

In [71]:
# Collect all tissue-level zarr paths
# tissue_out = Path("/Volumes/workl/Jona/spatial/second_run_output/tissues")

tissue_zarr_paths = sorted(tissue_out.glob("*.zarr"))
print("Found tissue-level zarrs:", len(tissue_zarr_paths))

tissue_sdatas = []
for p in tissue_zarr_paths:
    tissue_sdatas.append(spd.read_zarr(p))

# Concatenate
tissue_concat = spd.concatenate(tissue_sdatas, concatenate_tables=True)

print(tissue_concat)
tissue_concat_out = zarr_folder / "concatenated_tissues_sdata_second_run"
tissue_concat.write(tissue_concat_out, overwrite=True)


version mismatch: detected: RasterFormatV02, requested: FormatV04


Found tissue-level zarrs: 16


/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/janzules/miniforge3/envs/vishd_10x/lib/python3.11/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  co

SpatialData object
├── Images
│     ├── 'CyPSCA_1_1_hires_tissue_image': DataArray[cyx] (3, 2645, 2706)
│     ├── 'CyPSCA_1_2_hires_tissue_image': DataArray[cyx] (3, 2348, 2448)
│     ├── 'CyPSCA_2_4_hires_tissue_image': DataArray[cyx] (3, 2445, 2505)
│     ├── 'CyT72_1_2_hires_tissue_image': DataArray[cyx] (3, 2202, 2308)
│     ├── 'CyT72_1_4_hires_tissue_image': DataArray[cyx] (3, 2224, 2060)
│     ├── 'CyT72_2_3_hires_tissue_image': DataArray[cyx] (3, 2480, 2363)
│     ├── 'NoTx_1_4_hires_tissue_image': DataArray[cyx] (3, 2600, 1799)
│     ├── 'NoTx_2_2_hires_tissue_image': DataArray[cyx] (3, 2668, 2533)
│     ├── 'NoTx_2_4_hires_tissue_image': DataArray[cyx] (3, 2494, 2542)
│     ├── 'RTCyPSCA_1_3_hires_tissue_image': DataArray[cyx] (3, 2355, 2630)
│     ├── 'RTCyPSCA_1_4_hires_tissue_image': DataArray[cyx] (3, 2658, 2085)
│     ├── 'RTCyPSCA_2_3_hires_tissue_image': DataArray[cyx] (3, 2897, 2837)
│     ├── 'RTCyPSCA_2_4_hires_tissue_image': DataArray[cyx] (3, 2513, 2669)
│     ├──